# Tempo Run 2025 — Full Pipeline (Unsloth + Flash Attention 2)

Notebook tự chạy end-to-end: setup → download data → build SFT → balance → train Unsloth QLoRA → evaluate → submission.

## Điểm chính
- **Unsloth FastLanguageModel**: Triton kernels + FA2 tích hợp sẵn (trên Ampere+), nhanh ~2x TRL thuần
- **Flash Attention 2**: tăng tốc attention trên A100/Ampere+; tự động fallback SDPA trên T4x2 (Kaggle)
- **`enable_thinking=False`**: fix quan trọng cho Qwen3 — model mặc định ở thinking mode (sinh `<think>...` trước đáp án), cần tắt để xuất A/B/C/D trực tiếp, khớp giữa train và infer
- **Logits mode eval**: argmax over A/B/C/D token ids, không cần generate (nhanh 5-10x)
- **Code inline**: không import từ package `temprun`, tất cả logic viết trực tiếp trong notebook

## Prerequisites
- HF token (write scope): https://huggingface.co/settings/tokens
- HF private dataset repo có chứa file zip data (xem `COLAB_GUIDE.md` bước 0a)
- GPU: T4x2 (Kaggle, 2×16GB) hoặc A100 (Colab Pro+, 40GB+)

## Thời gian ước tính
| Bước | A100 | T4x2 |
|---|---|---|
| Setup + install | 3-5 min | 3-5 min |
| Download data | ~10s | ~10s |
| Build SFT + balance | ~10s | ~10s |
| Train Unsloth QLoRA | 20-40 min | 40-80 min |
| Evaluate | ~2 min | ~3-5 min |
| Submission | ~2 min | ~3-5 min |


In [ ]:
# 1. GPU check
import torch
assert torch.cuda.is_available(), 'C\u1ea7n GPU!'
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu_name}')
print(f'VRAM: {vram_gb:.1f} GB')
assert vram_gb >= 15, f'C\u00e2n \u226515GB VRAM (hi\u1ec7n c\u00f3 {vram_gb:.1f}GB)'

# Check SM version for Flash Attention 2 support
# FA2 requires Ampere+ (SM >= 8.0): A100, A10, 3090, 4090
# T4 (Turing, SM 7.5) does NOT support FA2 - will use SDPA instead
major, minor = torch.cuda.get_device_capability()
HAS_FA2 = major >= 8
print(f'SM version: {major}.{minor}')
print(f'Flash Attention 2: {"supported" if HAS_FA2 else "NOT supported (will use SDPA)"}')


In [ ]:
# 2. Clone repo (c\u1ea7n c\u00f3 th\u01b0 m\u1ee5c data/ + .env)
import os
REPO_URL = 'https://github.com/TristanDao/finetune_1B_MCQ_VN.git'
REPO_DIR = '/content/finetune_1B_MCQ_VN'
if os.path.isdir(REPO_DIR):
    %cd {REPO_DIR}
    !git pull --rebase --autostash
else:
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}


In [ ]:
# 3. C\u00e0i deps
# Unsloth: k\u00e9o s\u1eb5n torch, transformers, trl, peft, bitsandbytes, accelerate,
#           flash-attn (wheel prebuilt) v\u1edbi version t\u01b0\u01a1ng th\u00edch
!pip install -q unsloth

# Deps c\u00f2n l\u1ea1i cho data/eval (kh\u00f4ng b\u1ecb Unsloth bundle)
!pip install -q datasets sentencepiece scikit-learn pandas python-dotenv pyyaml huggingface-hub protobuf tqdm

# Flash Attention 2 cho eval/infer (Unsloth \u0111\u00e3 c\u00f3 s\u1eb5n trong training)
# L\u01b0u \u00fd: FA2 ch\u1ec9 h\u1ed7 tr\u1ee3 Ampere+ (A100/A10/3090/4090)
# T4x2 (Kaggle) kh\u00f4ng h\u1ed7 tr\u1ee3 FA2 \u2014 b\u1ecf qua d\u00f2ng n\u00e0y, cell 12 s\u1ebd t\u1ef1 \u0111\u1ed9ng d\u00f9ng SDPA
# A100 (Colab Pro+): b\u1ecf comment \u0111\u1ec3 c\u00e0i FA2 (build ~10-15 min)
# !pip install -q flash-attn --no-build-isolation


## 4. Write `.env`

Sửa các giá trị rồi chạy cell:


In [ ]:
%%writefile .env
HF_TOKEN=hf_your_token_here
HF_DATASET_REPO=ThinhDao/TempoRun2025_UIT
HF_DATASET_FILE=tempo-run-2025-run-with-ai-break-limits.zip


In [ ]:
# 5. Verify .env
import os
from dotenv import load_dotenv
load_dotenv(override=False)
required = ['HF_TOKEN', 'HF_DATASET_REPO', 'HF_DATASET_FILE']
ok = True
for k in required:
    v = os.environ.get(k, '')
    status = 'OK' if v and 'your' not in v else 'MISSING \u2014 edit .env!'
    if 'MISSING' in status:
        ok = False
    print(f'{k:20s} {status}')
assert ok, 'S\u1eeda .env r\u1ed3i ch\u1ea1y l\u1ea1i cell n\u00e0y'


## 6. Download data t\u1eeb HF private dataset

T\u1ea3i zip t\u1eeb Hugging Face, gi\u1ea3i n\u00e9n, flatten `Dataset/Dataset/{train,test}/` \u2192 `data/raw/{train,test}/`.


In [ ]:
# 6. Download data
import os, shutil, zipfile
from pathlib import Path
from huggingface_hub import hf_hub_download

REPO_DIR = Path('/content/finetune_1B_MCQ_VN')
DATA_RAW = REPO_DIR / 'data' / 'raw'
HF_REPO = os.environ['HF_DATASET_REPO']
HF_FILE = os.environ.get('HF_DATASET_FILE', 'tempo-run-2025-run-with-ai-break-limits.zip')
HF_TOKEN = os.environ['HF_TOKEN']

cache_dir = DATA_RAW.parent / '.hf_cache'
cache_dir.mkdir(parents=True, exist_ok=True)
print(f'Downloading {HF_FILE} from {HF_REPO}...')
cached = Path(hf_hub_download(
    repo_id=HF_REPO, filename=HF_FILE, repo_type='dataset',
    token=HF_TOKEN, local_dir=cache_dir,
))
print(f'Downloaded: {cached}')

# Extract + flatten
scratch = DATA_RAW.parent / '.data.scratch'
if scratch.exists():
    shutil.rmtree(scratch)
scratch.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(cached) as zf:
    zf.extractall(scratch)

SPLITS = ('train', 'test', 'public', 'private')
for split in SPLITS:
    src = scratch / 'Dataset' / 'Dataset' / split
    if not src.is_dir():
        src = scratch / split
    if not src.is_dir():
        continue
    dst = DATA_RAW / split
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    n = sum(1 for _ in dst.glob('*.json'))
    print(f'  {split}: {n} files -> {dst}')

# Copy helper files (sample_submission.csv etc.)
for f in scratch.glob('*'):
    if f.is_file() and f.suffix in {'.csv', '.md', '.txt'}:
        shutil.copy2(f, DATA_RAW / f.name)
        print(f'  helper: {f.name}')

shutil.rmtree(scratch, ignore_errors=True)
print('Done.')


## 7. Build SFT JSONL

Convert raw JSON \u2192 chat messages \u2192 train/eval JSONL v\u1edbi stratified split 90/10.
T\u1ea5t c\u1ea3 logic inline \u2014 kh\u00f4ng import t\u1eeb package `temprun`.


In [ ]:
# 7. Build SFT JSONL \u2014 inline t\u1ea5t c\u1ea3 logic
import json, glob, random
from pathlib import Path
from collections import Counter
from sklearn.model_selection import train_test_split

REPO_DIR = Path('/content/finetune_1B_MCQ_VN')
TRAIN_DIR = REPO_DIR / 'data' / 'raw' / 'train'
OUT_DIR = REPO_DIR / 'data' / 'processed'
OUT_DIR.mkdir(parents=True, exist_ok=True)

SYSTEM_PROMPT = 'B\u1ea1n l\u00e0 h\u1ec7 th\u1ed1ng tr\u1ea3 l\u1eddi tr\u1eafc nghi\u1ec7m. Ch\u1ec9 xu\u1ea5t duy nh\u1ea5t 1 k\u00fd t\u1ef1 A/B/C/D.'

def format_choices(choices):
    """Render choices deterministically A-D."""
    ordered = []
    for k in ['A', 'B', 'C', 'D']:
        if k in choices and choices[k] is not None:
            ordered.append(f'{k}: {choices[k]}')
    return '\n'.join(ordered)

def build_user_instruction(title, content, question, choices):
    """Build user-message text cho 1 MCQ item."""
    ct = format_choices(choices)
    return (
        'B\u1ea1n l\u00e0 h\u1ec7 th\u1ed1ng tr\u1ea3 l\u1eddi tr\u1eafc nghi\u1ec7m. H\u00e3y \u0111\u1ecdc v\u0103n b\u1ea3n v\u00e0 c\u00e2u h\u1ecfi, '
        'ch\u1ec9 ch\u1ecdn **m\u1ed9t \u0111\u00e1p \u00e1n duy nh\u1ea5t** t\u1eeb A/B/C/D, kh\u00f4ng gi\u1ea3i th\u00edch, kh\u00f4ng th\u00eam n\u1ed9i dung kh\u00e1c.\n\n'
        'V\u0103n b\u1ea3n:\n'
        f'Ti\u00eau \u0111\u1ec1: {title}\n\n'
        f'N\u1ed9i dung: {content}\n\n'
        'C\u00e2u h\u1ecfi:\n'
        f'{question}\n\n'
        'C\u00e1c l\u1ef1a ch\u1ecdn:\n'
        f'{ct}\n\n'
        'Ch\u1ec9 tr\u1ea3 l\u1eddi \u0111\u00fang 1 k\u00fd t\u1ef1: A, B, C ho\u1eb7c D.'
    )

def build_messages(system_prompt, user_instruction, assistant=None):
    msgs = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': user_instruction},
    ]
    if assistant is not None:
        msgs.append({'role': 'assistant', 'content': assistant})
    return msgs

# Load all JSON files and build rows
rows = []
dropped = 0
for path in sorted(glob.glob(str(TRAIN_DIR / '*.json'))):
    with open(path, encoding='utf-8') as f:
        d = json.load(f)
    content = (d.get('content:') or d.get('content') or '').strip()
    title = (d.get('title:') or d.get('title') or '').strip()
    questions = d.get('questions') or []
    if not content or not questions:
        dropped += 1
        continue
    for idx, q in enumerate(questions, start=1):
        question = (q.get('question') or '').strip()
        choices = q.get('choices') or {}
        raw_label = q.get('correct_answer')
        label = raw_label.strip().upper()[:1] if raw_label else None
        if not question or not choices:
            continue
        if label is not None and label not in {'A', 'B', 'C', 'D'}:
            continue
        user = build_user_instruction(title, content, question, choices)
        assistant = label if label is not None else None
        msgs = build_messages(SYSTEM_PROMPT, user, assistant=assistant)
        row = {
            'messages': msgs,
            'title': title, 'content': content,
            'question': question, 'choices': choices,
        }
        if label is not None:
            row['label'] = label
        rows.append(row)

print(f'Built {len(rows)} rows (dropped {dropped} docs)')

# Stratified split 90/10
labels = [r.get('label', 'UNK') for r in rows]
train_rows, eval_rows = train_test_split(
    rows, test_size=0.1, random_state=3407, shuffle=True, stratify=labels,
)
print(f'Train: {len(train_rows)}  Eval: {len(eval_rows)}')
print(f'Train label dist: {dict(Counter(r.get("label","UNK") for r in train_rows))}')
print(f'Eval  label dist: {dict(Counter(r.get("label","UNK") for r in eval_rows))}')

def write_jsonl(data, path):
    with open(path, 'w', encoding='utf-8') as f:
        for r in data:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')

def read_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

write_jsonl(train_rows, OUT_DIR / 'train.jsonl')
write_jsonl(eval_rows, OUT_DIR / 'eval.jsonl')
print(f'Wrote {OUT_DIR / "train.jsonl"} and {OUT_DIR / "eval.jsonl"}')


## 8. Balance labels (rotate choices)

Data g\u1ed1c bias m\u1ea1nh (vd D ch\u1ec9 ~3%). Xoay `choices` sao cho \u0111\u00e1p \u00e1n \u0111\u00fang cycles A\u2192B\u2192C\u2192D \u2192 distribution \u02241:1:1:1.
Kh\u00f4ng g\u1ecdi API, ch\u1ea1y trong ~1 gi\u00e2y.


In [ ]:
# 8. Balance labels \u2014 inline balance_via_reorder + merge
# Reuse: build_user_instruction, build_messages, write_jsonl, read_jsonl,
#         train_test_split from cell 7
from collections import Counter
from pathlib import Path
import random

REPO_DIR = Path('/content/finetune_1B_MCQ_VN')
PROC_DIR = REPO_DIR / 'data' / 'processed'
FINAL_DIR = PROC_DIR / 'final'
FINAL_DIR.mkdir(parents=True, exist_ok=True)

def reorder_row_for_label(row, target_label):
    """Rotate choices sao cho correct answer lands on target_label."""
    orig_label = row.get('label', '')
    if orig_label not in {'A','B','C','D'} or target_label not in {'A','B','C','D'}:
        return row
    choices = row.get('choices') or {}
    if not all(k in choices for k in ('A','B','C','D')):
        return row
    shift = (ord(target_label) - ord(orig_label)) % 4
    if shift == 0:
        return row
    new_choices = {}
    for k, v in choices.items():
        if k in {'A','B','C','D'}:
            new_choices[chr(ord('A') + (ord(k) - ord('A') + shift) % 4)] = v
        else:
            new_choices[k] = v
    new_user = build_user_instruction(
        row.get('title',''), row.get('content',''),
        row.get('question',''), new_choices,
    )
    new_msgs = build_messages(SYSTEM_PROMPT, new_user, assistant=target_label)
    new_row = dict(row)
    new_row['messages'] = new_msgs
    new_row['choices'] = new_choices
    new_row['label'] = target_label
    return new_row

def balance_via_reorder(rows, seed=3407):
    """Rebalance by rotating choices so label cycles A->B->C->D."""
    rng = random.Random(seed)
    indices = list(range(len(rows)))
    rng.shuffle(indices)
    target_labels = ['A', 'B', 'C', 'D']
    return [reorder_row_for_label(rows[i], target_labels[i % 4]) for i in indices]

# Load train.jsonl, balance, merge with eval, re-split
rows = read_jsonl(PROC_DIR / 'train.jsonl')
pre_dist = dict(Counter(r.get('label','UNK') for r in rows))
print(f'Before balance: {pre_dist}')

balanced = balance_via_reorder(rows, seed=3407)
post_dist = dict(Counter(r.get('label','UNK') for r in balanced))
print(f'After balance:  {post_dist}')

eval_rows = read_jsonl(PROC_DIR / 'eval.jsonl')
all_rows = balanced + eval_rows
print(f'Total: {len(all_rows)} (balanced {len(balanced)} + eval {len(eval_rows)})')

labels = [r.get('label','UNK') for r in all_rows]
final_train, final_eval = train_test_split(
    all_rows, test_size=0.1, random_state=3407, shuffle=True, stratify=labels,
)
print(f'Final train: {len(final_train)}  eval: {len(final_eval)}')

write_jsonl(final_train, FINAL_DIR / 'train.jsonl')
write_jsonl(final_eval, FINAL_DIR / 'eval.jsonl')
print(f'Wrote {FINAL_DIR}')


## 9. Load Unsloth model + attach LoRA

Unsloth `FastLanguageModel` t\u1ef1 x\u00e0i Triton kernels + **FA2 t\u00edch h\u1ee3p s\u1eb5n** trong training.
Kh\u00f4ng c\u1ea7n `pip install flash-attn` ri\u00eang cho training.


In [ ]:
# 9. Load Unsloth model + LoRA
from unsloth import FastLanguageModel
import torch

MODEL_NAME = 'Qwen/Qwen3-0.6B'
MAX_SEQ_LENGTH = 2048
LORA_R = 32
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
TARGET_MODULES = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']

# Load model + tokenizer \u2014 Unsloth t\u1ef1 x\u1eed l\u00fd FA2 + Triton kernels
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,  # auto: bf16 tr\u00ean A100, fp16 tr\u00ean cũ h\u01a1n
    load_in_4bit=True,
)
print(f'Loaded {MODEL_NAME}, vocab={tokenizer.vocab_size}')

# Attach LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=3407,
    max_seq_length=MAX_SEQ_LENGTH,
)
model.print_trainable_parameters()


## 10. Render chat text \u2014 fix `enable_thinking=False`

**Quan tr\u1ecdng nh\u1ea5t**: Qwen3 chat template c\u00f3 thinking mode m\u1eb7c \u0111\u1ecbnh.
Khi `add_generation_prompt=True` m\u00e0 KH\u00d4NG set `enable_thinking=False`,
template ch\u1ec9 th\u00eam `<|im_start|>assistant\n` \u2192 model s\u1ebd sinh `\u003cthink\u003e...` tr\u01b0\u1edbc \u0111\u00e1p \u00e1n.

Khi `enable_thinking=False`, template th\u00eam `<|im_start|>assistant\n\n\n` (empty thinking block)
\u2192 model xu\u1ea5t A/B/C/D tr\u1ef1c ti\u1ebfp. **Train v\u00e0 infer ph\u1ea3i d\u00f9ng c\u00f9ng prefix n\u00e0y.**


In [ ]:
# 10. Render chat text \u2014 demo fix thinking mode
# Reuse: train_rows from cell 7

sample = final_train[0]  # t\u1eeb cell 8
msgs = sample['messages']
print('=== Messages ===')
for m in msgs:
    print(f'  [{m["role"]}] {m["content"][:80]}...')

prefix_msgs = [m for m in msgs if m['role'] != 'assistant']

# C\u00e1ch C\u0168 (kh\u00f4ng fix) \u2014 model mu\u1ed1n sinh <think>
old_render = tokenizer.apply_chat_template(
    prefix_msgs, tokenize=False, add_generation_prompt=True,
)
print('\n=== OLD (enable_thinking m\u1eb7c \u0111\u1ecbnh = True) ===')
print(repr(old_render[-80:]))

# C\u00e1ch M\u1edaI (fix) \u2014 enable_thinking=False
new_render = tokenizer.apply_chat_template(
    prefix_msgs, tokenize=False, add_generation_prompt=True,
    enable_thinking=False,
)
print('\n=== NEW (enable_thinking=False) ===')
print(repr(new_render[-80:]))

# Render cho training: prefix + assistant content + EOS
def render_chat_for_training(tokenizer, messages, enable_thinking=False):
    """Render messages th\u00e0nh 1 string cho SFT.
    T\u00e1ch assistant message, render prefix v\u1edbi add_generation_prompt=True,
    r\u1ed3i append assistant_content + eos.
    """
    assistant_content = None
    prefix = []
    for m in messages:
        if m['role'] == 'assistant':
            assistant_content = m['content']
        else:
            prefix.append(m)
    text = tokenizer.apply_chat_template(
        prefix, tokenize=False, add_generation_prompt=True,
        enable_thinking=enable_thinking,
    )
    if assistant_content is not None:
        text += assistant_content
    eos = tokenizer.eos_token
    if eos and not text.endswith(eos):
        text += eos
    return text

train_text = render_chat_for_training(tokenizer, msgs, enable_thinking=False)
print('\n=== Training text (enable_thinking=False) ===')
print(repr(train_text[-120:]))


## 11. Train Unsloth QLoRA

- `completion_only_loss=True`: loss ch\u1ec9 tr\u00ean assistant token (A/B/C/D), kh\u00f4ng t\u00ednh tr\u00ean prompt
- `packing=True`: g\u1ed9p nhi\u1ec1u samples v\u00e0o 1 sequence \u2192 t\u0103ng throughput
- Effective batch = 8 \u00d7 4 = 32
- 3 epochs, LR 2e-4, cosine schedule

Th\u1eddi gian: ~20-40 min tr\u00ean A100 cho ~4000 samples.


In [ ]:
# 11. Train Unsloth QLoRA
import json
from pathlib import Path
from datasets import Dataset
from trl import SFTConfig, SFTTrainer

# Reuse: render_chat_for_training, read_jsonl from cell 10 / cell 7

REPO_DIR = Path('/content/finetune_1B_MCQ_VN')
TRAIN_JSONL = REPO_DIR / 'data' / 'processed' / 'final' / 'train.jsonl'
EVAL_JSONL = REPO_DIR / 'data' / 'processed' / 'final' / 'eval.jsonl'
OUTPUT_DIR = REPO_DIR / 'artifacts' / 'unsloth_qwen3_0_6b'

train_rows = read_jsonl(TRAIN_JSONL)
eval_rows = read_jsonl(EVAL_JSONL)
print(f'Train: {len(train_rows)}  Eval: {len(eval_rows)}')

# Render to text with enable_thinking=False
def to_text(row):
    return {'text': render_chat_for_training(tokenizer, row['messages'], enable_thinking=False)}

train_ds = Dataset.from_list(train_rows).map(
    to_text, remove_columns=list(train_rows[0].keys()),
)
eval_ds = Dataset.from_list(eval_rows).map(
    to_text, remove_columns=list(eval_rows[0].keys()),
)
print(f'Dataset: train={len(train_ds)} eval={len(eval_ds)}')
print(f'Sample text (last 100 chars): {repr(train_ds[0]["text"][-100:])}')

# SFT Config
sft_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    # T4x2 (Kaggle): n\u1ebfu OOM, gi\u1ea3m xu\u1ed1ng 4 v\u00e0 t\u0103ng gradient_accumulation_steps l\u00ean 8
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,  # effective batch = 32
    num_train_epochs=3,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.06,
    lr_scheduler_type='cosine',
    logging_steps=20,
    eval_steps=200,
    save_steps=200,
    save_total_limit=2,
    max_grad_norm=0.3,
    optim='adamw_8bit',
    bf16=True,
    gradient_checkpointing=True,
    dataset_text_field='text',
    packing=True,
    max_length=MAX_SEQ_LENGTH,
    eval_strategy='steps',
    save_strategy='steps',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    completion_only_loss=True,  # loss ch\u1ec9 tr\u00ean assistant token
    report_to=[],
    seed=3407,
)

trainer = SFTTrainer(
    model=model,
    args=sft_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
)

print('Starting training...')
trainer.train()

# Save LoRA adapter
adapter_dir = OUTPUT_DIR / 'adapter'
adapter_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))
print(f'Saved adapter to {adapter_dir}')

# Save merged 16-bit (optional, cho standalone inference)
merged_dir = OUTPUT_DIR / 'merged_16bit'
merged_dir.mkdir(parents=True, exist_ok=True)
try:
    model.save_pretrained_merged(str(merged_dir), tokenizer, save_method='merged_16bit')
    print(f'Saved merged 16-bit to {merged_dir}')
except Exception as e:
    print(f'save_pretrained_merged failed: {e}')


## 12. Evaluate (logits mode)

Load model + LoRA adapter, merge, r\u1ed3i argmax over A/B/C/D token ids.
Th\u1eed `flash_attention_2` cho eval, fallback `sdpa` n\u1ebfu flash-attn chưa c\u00e0i ri\u00eang.


In [ ]:
# 12. Evaluate \u2014 logits mode (argmax over A/B/C/D)
import torch
from pathlib import Path
from collections import Counter, defaultdict
from tqdm import tqdm
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Reuse: read_jsonl from cell 7

REPO_DIR = Path('/content/finetune_1B_MCQ_VN')
ADAPTER_PATH = REPO_DIR / 'artifacts' / 'unsloth_qwen3_0_6b' / 'adapter'
EVAL_JSONL = REPO_DIR / 'data' / 'processed' / 'final' / 'eval.jsonl'
MODEL_NAME = 'Qwen/Qwen3-0.6B'
MAX_LENGTH = 2048
BATCH_SIZE = 16
SYSTEM_PROMPT = 'B\u1ea1n l\u00e0 h\u1ec7 th\u1ed1ng tr\u1ea3 l\u1eddi tr\u1eafc nghi\u1ec7m. Ch\u1ec9 xu\u1ea5t duy nh\u1ea5t 1 k\u00fd t\u1ef1 A/B/C/D.'

# Load model for inference \u2014 use FA2 if supported, else SDPA
bf16 = torch.cuda.is_bf16_supported()
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16 if bf16 else torch.float16,
    bnb_4bit_use_double_quant=True,
)
attn_impl = 'flash_attention_2' if HAS_FA2 else 'sdpa'
print(f'Using attention: {attn_impl}')
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, device_map='auto',
    torch_dtype=torch.bfloat16 if bf16 else torch.float16,
    attn_implementation=attn_impl,
    trust_remote_code=True, quantization_config=bnb,
)

model = PeftModel.from_pretrained(base_model, str(ADAPTER_PATH))
model = model.merge_and_unload()
model.eval()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Get token IDs for A/B/C/D
letter_ids = {}
for ch in ['A', 'B', 'C', 'D']:
    ids = tokenizer(ch, add_special_tokens=False).input_ids
    assert len(ids) == 1, f"Letter '{ch}' is not single token (got {ids})"
    letter_ids[ch] = ids[0]
print(f'Letter token IDs: {letter_ids}')

# Build prompts with enable_thinking=False (kh\u1edbp v\u1edbi train)
def to_chat_prompt(tokenizer, instruction, system_prompt):
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': instruction},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
        enable_thinking=False,
    )

eval_rows = read_jsonl(EVAL_JSONL)
print(f'Eval rows: {len(eval_rows)}')

prompts = []
labels = []
for r in eval_rows:
    instr = r['messages'][1]['content']  # user message
    prompts.append(to_chat_prompt(tokenizer, instr, SYSTEM_PROMPT))
    labels.append(r['label'])

# Eval: argmax over A/B/C/D logits
correct = 0
preds = []
with torch.no_grad():
    for i in tqdm(range(0, len(prompts), BATCH_SIZE), desc='Evaluating'):
        ps = prompts[i:i+BATCH_SIZE]
        ls = labels[i:i+BATCH_SIZE]
        enc = tokenizer(ps, return_tensors='pt', padding=True,
                        truncation=True, max_length=MAX_LENGTH).to(model.device)
        logits = model(**enc).logits
        last_pos = enc.attention_mask.sum(dim=1) - 1
        batch_idx = torch.arange(logits.size(0), device=logits.device)
        next_logits = logits[batch_idx, last_pos, :]
        cand_ids = torch.tensor(list(letter_ids.values()), device=logits.device)
        cand_logits = next_logits[:, cand_ids]
        pred_idx = cand_logits.argmax(dim=1)
        idx2letter = list(letter_ids.keys())
        batch_preds = [idx2letter[i.item()] for i in pred_idx]
        preds.extend(batch_preds)
        for p, g in zip(batch_preds, ls):
            correct += int(p == g)

acc = 100.0 * correct / len(eval_rows)
print(f'\n=== EVAL RESULTS ===')
print(f'Total: {len(eval_rows)}')
print(f'Correct: {correct}')
print(f'Accuracy: {acc:.2f}%')

# Confusion matrix
cm = defaultdict(lambda: Counter())
for p, g in zip(preds, labels):
    cm[g][p] += 1
letters_sorted = sorted(set(cm.keys()) | set(preds))
print(f'\nPred dist: {dict(Counter(preds))}')
print('Confusion (rows=label, cols=pred):')
header = '     ' + '  '.join(f'{c:>5}' for c in letters_sorted)
print(header)
for lab in letters_sorted:
    if lab not in cm:
        continue
    row = [cm[lab][p] for p in letters_sorted]
    print(f'{lab:>3}: ' + '  '.join(f'{n:>5}' for n in row))


## 13. Generate submission CSV

D\u00f9ng model \u0111\u00e3 load \u1edf cell 12, predict cho t\u1ea5t c\u1ea3 test items, ghi `submissions/sub_unsloth.csv`.


In [ ]:
# 13. Generate submission
import json, glob
import pandas as pd
from pathlib import Path
from tqdm import tqdm

# Reuse: to_chat_prompt, letter_ids, model, tokenizer, read_jsonl from cell 12
#         build_user_instruction from cell 7

REPO_DIR = Path('/content/finetune_1B_MCQ_VN')
TEST_DIR = REPO_DIR / 'data' / 'raw' / 'test'
OUT_CSV = REPO_DIR / 'submissions' / 'sub_unsloth.csv'
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
MAX_LENGTH = 2048
BATCH_SIZE = 16
SYSTEM_PROMPT = 'B\u1ea1n l\u00e0 h\u1ec7 th\u1ed1ng tr\u1ea3 l\u1eddi tr\u1eafc nghi\u1ec7m. Ch\u1ec9 xu\u1ea5t duy nh\u1ea5t 1 k\u00fd t\u1ef1 A/B/C/D.'

# Walk test JSON files \u2014 same logic as infer.py
items = []
for path in sorted(glob.glob(str(TEST_DIR / '*.json'))):
    article_id = Path(path).stem
    with open(path, encoding='utf-8') as f:
        d = json.load(f)
    content = (d.get('content:') or d.get('content') or '').strip()
    title = (d.get('title:') or d.get('title') or '').strip()
    questions = d.get('questions') or []
    if not content or not questions:
        continue
    for idx, q in enumerate(questions, start=1):
        question = (q.get('question') or '').strip()
        choices = q.get('choices') or {}
        if not question or not choices:
            continue
        items.append({
            'article_id': article_id,
            'qid': idx,
            'row_id': f'{article_id}__q{idx}',
            'title': title,
            'content': content,
            'question': question,
            'choices': choices,
        })

print(f'Test items: {len(items)}')

# Build prompts
prompts = []
for it in items:
    instr = build_user_instruction(it['title'], it['content'], it['question'], it['choices'])
    prompts.append(to_chat_prompt(tokenizer, instr, SYSTEM_PROMPT))

# Predict \u2014 same logits mode as eval
results = []
with torch.no_grad():
    for i in tqdm(range(0, len(prompts), BATCH_SIZE), desc='Inferring'):
        ps = prompts[i:i+BATCH_SIZE]
        enc = tokenizer(ps, return_tensors='pt', padding=True,
                        truncation=True, max_length=MAX_LENGTH).to(model.device)
        logits = model(**enc).logits
        last_pos = enc.attention_mask.sum(dim=1) - 1
        batch_idx = torch.arange(logits.size(0), device=logits.device)
        next_logits = logits[batch_idx, last_pos, :]
        cand_ids = torch.tensor(list(letter_ids.values()), device=logits.device)
        cand_logits = next_logits[:, cand_ids]
        pred_idx = cand_logits.argmax(dim=1)
        idx2letter = list(letter_ids.keys())
        for j, p in enumerate(pred_idx):
            results.append({
                'row_id': items[i + j]['row_id'],
                'answer': idx2letter[p.item()],
            })

df = pd.DataFrame(results)
df = df[['row_id', 'answer']]
df.to_csv(OUT_CSV, index=False)
print(f'\nWrote {len(df)} rows to {OUT_CSV}')
print(f'Pred dist: {df["answer"].value_counts().to_dict()}')
print(f'Duplicates: {df["row_id"].duplicated().sum()}')
print(df.head(10))


## Summary

Pipeline ho\u00e0n t\u1ea5t! Artifacts:
- **Adapter**: `artifacts/unsloth_qwen3_0_6b/adapter/`
- **Merged 16-bit**: `artifacts/unsloth_qwen3_0_6b/merged_16bit/` (n\u1ebfu save th\u00e0nh c\u00f4ng)
- **Eval accuracy**: in cell 12
- **Submission**: `submissions/sub_unsloth.csv`

## Next steps
- **Push l\u00ean HF Hub**: xem `notebooks/07_push_hf.ipynb`
- **So s\u00e1nh v\u1edbi TRL**: ch\u1ea1y `python scripts/train.py --config configs/qlora_qwen3_0_6b.yaml` r\u1ed3i eval
- **Backup l\u00ean Drive**: copy `artifacts/unsloth_qwen3_0_6b/` sang Google Drive
- **Improve**: th\u1eed enrich data (paraphrase/synth qua DashScope) \u2014 xem `notebooks/02_enrich.ipynb`
- **Thay model**: \u0111\u1ed5i `MODEL_NAME` \u1edf cell 9 th\u00e0m `Qwen/Qwen3.5-0.8B` (c\u1ea7n transformers@main + x\u1eed l\u00fd multimodal)
